## Imports & Paths

In [ ]:
from pathlib import Path
import pandas as pd
import torch
from sentence_transformers import SentenceTransformer

# ====== BASE PATHS (using relative paths for GitHub) ======
# Assuming this notebook is in: Retrieval-Augmented-Classification/embedding_creation/
# And folds are in: Data-and-preprocess/processed/folds/

BASE_DIR   = Path(".").resolve().parent.parent  # Go up to root
FOLDS_DIR  = BASE_DIR / "Data-and-preprocess" / "processed" / "folds"
OUT_DIR    = Path(".").resolve().parent / "embeddings"  # Current dir's parent
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Folds to process (train split only)
FOLDS = [1, 2, 3, 4, 5]

# ====== MODEL CONFIG ======
MODEL_NAME = "intfloat/multilingual-e5-base"  # Hebrew-friendly encoders
BATCH_SIZE = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("FOLDS_DIR:", FOLDS_DIR)
print("OUT_DIR:", OUT_DIR)
print("DEVICE:", DEVICE)
print("Files available:", list(FOLDS_DIR.glob("fold_*_train.csv"))[:2] if FOLDS_DIR.exists() else "Path not found")

## Load Model & Helper Functions

In [ ]:
import json, math
from typing import List, Any

# ====== Load model ======
model = SentenceTransformer(MODEL_NAME, device=DEVICE)
model.max_seq_length = 512  # keep reasonably short for speed

# ====== Helpers ======
def _as_text(x: Any) -> str:
    s = "" if x is None else str(x)
    return "" if s == "nan" else s

def _norm(v: List[float]) -> float:
    return math.sqrt(sum(x*x for x in v)) if v else 0.0

def _to_json(x: Any) -> str:
    try:
        return json.dumps(x, ensure_ascii=False)
    except Exception:
        return "[]"

def embed_texts(texts: List[str], batch_size: int = BATCH_SIZE) -> List[List[float]]:
    """Embed Hebrew text locally using E5 model."""
    if not texts:
        return []
    # E5 prefers "passage: " prefix for corpus embeddings
    prefixed = [f"passage: {t}" if t else "passage: " for t in texts]
    embs = model.encode(
        prefixed,
        convert_to_numpy=True,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=False,  # store raw vectors (we'll L2 later in retriever)
    )
    return [row.tolist() for row in embs]

## Embedding Function for Single Fold

In [ ]:
def embed_fold_train(fold_id: int):
    """Embed question and answer texts for a single fold's training set."""
    train_csv = FOLDS_DIR / f"fold_{fold_id}_train.csv"
    if not train_csv.exists():
        raise FileNotFoundError(f"Missing train CSV for fold {fold_id}: {train_csv}")

    df = pd.read_csv(train_csv, encoding="utf-8")
    required = ["question", "answer", "final_decision", "final_indicators"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Fold {fold_id} train is missing columns: {missing}")

    # Stable id: prefer existing 'idx' or 'id', else create one
    if "id" in df.columns:
        id_col = "id"
    elif "idx" in df.columns:
        id_col = "idx"
        df = df.rename(columns={"idx": "id"})
    else:
        df = df.reset_index().rename(columns={"index": "id"})
        id_col = "id"

    Q = [_as_text(x) for x in df["question"].tolist()]
    A = [_as_text(x) for x in df["answer"].tolist()]

    print(f"[Fold {fold_id}] Embedding {len(df)} train rows with {MODEL_NAME} on {DEVICE} …")
    q_vecs = embed_texts(Q)
    a_vecs = embed_texts(A)

    out = pd.DataFrame({
        "id": df["id"],
        "question": df["question"],
        "answer": df["answer"],
        "final_decision": df["final_decision"],
        "final_indicators": df["final_indicators"],
    })
    out["q_vec"]  = q_vecs
    out["a_vec"]  = a_vecs
    out["q_norm"] = out["q_vec"].apply(_norm)
    out["a_norm"] = out["a_vec"].apply(_norm)

    # Paths per fold
    fold_dir = OUT_DIR / f"fold_{fold_id}"
    fold_dir.mkdir(parents=True, exist_ok=True)
    out_parquet = fold_dir / f"fold_{fold_id}_train_embeddings.parquet"
    out_csv     = fold_dir / f"fold_{fold_id}_train_embeddings.csv"
    rag_copy    = fold_dir / f"fold_{fold_id}_train_RAG_dataset.csv"

    # Save Parquet (keeps vectors)
    out.to_parquet(out_parquet, index=False)
    
    # Save CSV (JSON-encode vectors for readability)
    out_csv_df = out.copy()
    out_csv_df["q_vec"] = out_csv_df["q_vec"].apply(_to_json)
    out_csv_df["a_vec"] = out_csv_df["a_vec"].apply(_to_json)
    out_csv_df.to_csv(out_csv, index=False, encoding="utf-8")

    # Also store a clean copy of the train dataset the retriever will index
    df.to_csv(rag_copy, index=False, encoding="utf-8")

    print(f"[Fold {fold_id}] ✅ Saved Parquet → {out_parquet}")
    print(f"[Fold {fold_id}] ✅ Saved CSV     → {out_csv}")
    print(f"[Fold {fold_id}] ✅ Saved RAG set → {rag_copy}")

    # Preview
    display(out.head(5)[["id","question","answer","final_decision","final_indicators","q_norm","a_norm"]])

## Run Embeddings for All Folds

In [ ]:
for f in FOLDS:
    embed_fold_train(f)
print("✅ Done embedding all folds.")